<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/07_temas_actuales/71_transformers_y_llm.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Transformers, GPT y modelos de lenguaje

**Pregunta guía:** ¿Cómo predice el siguiente token un transformer causal?<br>
**Duración sugerida:** 6 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere PyTorch; GPU opcional.** Este notebook construye un GPT
diminuto de caracteres. Busca revelar los componentes, no reproducir la
escala ni las capacidades de un sistema comercial.

Flujo conceptual:

1. tokenizar texto y convertir tokens en embeddings;
2. mezclar contexto con autoatención causal;
3. preentrenar mediante predicción del siguiente token;
4. en sistemas asistentes, realizar postentrenamiento para seguir
   instrucciones y preferencias;
5. durante inferencia, muestrear tokens autoregresivamente y, en algunos
   sistemas, coordinar herramientas externas.


## Atención escalada

Desde una secuencia $X$ calculamos $Q=XW_Q$, $K=XW_K$, $V=XW_V$:

$$\mathrm{Attention}(Q,K,V)=
\mathrm{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V.$$

La máscara causal $M_{ij}=-\infty$ si $j>i$: la posición $i$ no puede
mirar tokens futuros. Varias cabezas aprenden proyecciones distintas y
concatenan sus resultados. La atención por sí sola no conoce el orden;
añadimos embeddings posicionales.


In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

def softmax_estable(x,axis=-1):
    e=np.exp(x-np.max(x,axis=axis,keepdims=True)); return e/e.sum(axis=axis,keepdims=True)

rng=np.random.default_rng(42); X_demo=rng.normal(size=(5,4))
Q=X_demo@rng.normal(size=(4,4)); K=X_demo@rng.normal(size=(4,4)); V=X_demo@rng.normal(size=(4,4))
scores=Q@K.T/np.sqrt(4); scores[np.triu_indices(5,k=1)]=-np.inf
A=softmax_estable(scores); contexto=A@V
print("filas suman",A.sum(axis=1)); print("contexto",contexto.shape)
plt.imshow(A,cmap="viridis",vmin=0,vmax=1); plt.xlabel("token consultado"); plt.ylabel("token que consulta"); plt.colorbar(); plt.show()


## Objetivo autoregresivo

Dada una secuencia $x_1,\ldots,x_T$, maximizamos
$\sum_t\log p_\theta(x_t\mid x_{<t})$. La entropía cruzada compara los
logits del vocabulario con el siguiente token. *Teacher forcing* permite
calcular todas las posiciones en paralelo durante entrenamiento; generar
sigue siendo secuencial.

El corpus siguiente es deliberadamente pequeño y escrito para esta
práctica. Un modelo real necesita mucha más diversidad, cómputo,
evaluación, documentación de datos y salvaguardas.


In [ ]:
SEMILLA=42; torch.manual_seed(SEMILLA)
dispositivo=torch.device("cuda" if torch.cuda.is_available() else "cpu")
base=("la energia se conserva en un sistema aislado. "
      "la fuerza cambia el momento. la luz transporta energia y momento. "
      "un modelo aproxima datos dentro de supuestos. medir implica incertidumbre.\n")
texto=base*220
caracteres=sorted(set(texto)); stoi={c:i for i,c in enumerate(caracteres)}; itos={i:c for c,i in stoi.items()}
datos=torch.tensor([stoi[c] for c in texto],dtype=torch.long)
corte=int(.9*len(datos)); datos_train,datos_val=datos[:corte],datos[corte:]
VOCAB=len(caracteres); BLOQUE=48; BATCH=64

def lote(fuente):
    i=torch.randint(len(fuente)-BLOQUE-1,(BATCH,))
    x=torch.stack([fuente[j:j+BLOQUE] for j in i]); y=torch.stack([fuente[j+1:j+BLOQUE+1] for j in i])
    return x.to(dispositivo),y.to(dispositivo)
print("vocabulario",VOCAB,"tokens",len(datos),"dispositivo",dispositivo)


In [ ]:
class AtenciónCausal(nn.Module):
    def __init__(self,dimensión=64,cabezas=4):
        super().__init__(); self.cabezas=cabezas; self.dk=dimensión//cabezas
        self.qkv=nn.Linear(dimensión,3*dimensión); self.proyección=nn.Linear(dimensión,dimensión)
        self.register_buffer("máscara",torch.tril(torch.ones(BLOQUE,BLOQUE,dtype=torch.bool)))
    def forward(self,x):
        B,T,C=x.shape
        qkv=self.qkv(x).reshape(B,T,3,self.cabezas,self.dk).permute(2,0,3,1,4)
        q,k,v=qkv[0],qkv[1],qkv[2]
        pesos=(q@k.transpose(-2,-1))/math.sqrt(self.dk)
        pesos=pesos.masked_fill(~self.máscara[:T,:T],float("-inf")); pesos=torch.softmax(pesos,dim=-1)
        salida=(pesos@v).transpose(1,2).contiguous().reshape(B,T,C)
        return self.proyección(salida)

class BloqueTransformer(nn.Module):
    def __init__(self,dimensión=64,cabezas=4):
        super().__init__(); self.ln1=nn.LayerNorm(dimensión); self.ln2=nn.LayerNorm(dimensión)
        self.atención=AtenciónCausal(dimensión,cabezas)
        self.mlp=nn.Sequential(nn.Linear(dimensión,4*dimensión),nn.GELU(),nn.Linear(4*dimensión,dimensión))
    def forward(self,x):
        x=x+self.atención(self.ln1(x)); return x+self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self,dimensión=64,capas=2,cabezas=4):
        super().__init__(); self.token=nn.Embedding(VOCAB,dimensión); self.pos=nn.Embedding(BLOQUE,dimensión)
        self.bloques=nn.Sequential(*[BloqueTransformer(dimensión,cabezas) for _ in range(capas)])
        self.ln=nn.LayerNorm(dimensión); self.cabeza=nn.Linear(dimensión,VOCAB,bias=False)
    def forward(self,idx,objetivo=None):
        T=idx.shape[1]; x=self.token(idx)+self.pos(torch.arange(T,device=idx.device))
        logits=self.cabeza(self.ln(self.bloques(x)))
        pérdida=None if objetivo is None else F.cross_entropy(logits.reshape(-1,VOCAB),objetivo.reshape(-1))
        return logits,pérdida
    @torch.no_grad()
    def generar(self,idx,nuevos=180,temperatura=.8):
        for _ in range(nuevos):
            logits,_=self(idx[:,-BLOQUE:]); prob=torch.softmax(logits[:,-1]/temperatura,dim=-1)
            idx=torch.cat([idx,torch.multinomial(prob,1)],dim=1)
        return idx


In [ ]:
modelo=MiniGPT().to(dispositivo); optimizador=torch.optim.AdamW(modelo.parameters(),lr=3e-3,weight_decay=.01)
print("parámetros",sum(p.numel() for p in modelo.parameters()))
historial=[]
for paso in range(700):
    xb,yb=lote(datos_train); optimizador.zero_grad(); _,pérdida=modelo(xb,yb); pérdida.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(),1.0); optimizador.step()
    if paso%100==0:
        modelo.eval()
        with torch.no_grad(): xv,yv=lote(datos_val); val=modelo(xv,yv)[1].item()
        modelo.train(); historial.append((paso,pérdida.item(),val)); print(historial[-1])


In [ ]:
modelo.eval()
inicio=torch.tensor([[stoi["l"]]],device=dispositivo)
generado=modelo.generar(inicio,nuevos=240,temperatura=.75)[0].cpu().tolist()
print("".join(itos[i] for i in generado))
h=np.asarray(historial); plt.plot(h[:,0],h[:,1],label="train"); plt.plot(h[:,0],h[:,2],label="validation"); plt.xlabel("paso"); plt.ylabel("cross entropy"); plt.legend(); plt.show()


## Qué explica esto sobre asistentes actuales

El experimento sí ilustra tokenización, embeddings, atención causal,
bloques residuales, predicción autoregresiva, temperatura y
preentrenamiento. Sistemas actuales pueden además aceptar varias
modalidades, usar ventanas de contexto grandes, razonamiento especializado
y herramientas; las capacidades publicadas deben consultarse en la
[documentación oficial de modelos de OpenAI](https://developers.openai.com/api/docs/models).

No se deben inventar detalles sobre un producto concreto. El número de
parámetros, mezcla exacta de datos, arquitectura completa, postentrenamiento,
infraestructura de inferencia y reglas internas pueden no ser públicos.
Una explicación responsable separa el mecanismo académico conocido de
aquello que la organización realmente documenta.

Lecturas primarias: [Transformer](https://arxiv.org/abs/1706.03762),
[GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf),
[GPT-3](https://arxiv.org/abs/2005.14165) e
[InstructGPT](https://arxiv.org/abs/2203.02155).


**Ejercicios:** cambie longitud de contexto y mida validación; elimine
posiciones; visualice una cabeza; compare greedy, temperatura y top-k;
calcule costo $O(T^2)$ de la matriz de atención; documente sesgos del
corpus y proponga evaluaciones de factualidad, seguridad y memorization.
